# 🎬 Notebook 01 — Frame Extraction
**Goal:** Extract frames from raw exam videos and organise them into class folders.

| Setting | Value |
|---|---|
| Output | `/content/drive/MyDrive/dataset_frames/{cheating,not_cheating}/` |
| Frame skip | Every 5th frame (configurable) |
| Blur filter | Laplacian variance ≥ 80 (removes blurry frames) |
| Format | JPEG, quality 95 |

➡ **Next:** Run Notebook 02 to clean and split the dataset.

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 0 — Imports & Drive Mount
# ════════════════════════════════════════════════════════════════════

import os, cv2, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# ╔══════════════════════════════════════════════════════════╗
# ║          PROJECT-WIDE CONSTANTS — do not change          ║
# ║  These must be identical across all notebooks & scripts  ║
# ╚══════════════════════════════════════════════════════════╝
DATASET_PATH   = '/content/drive/MyDrive/dataset_final'   # split dataset
FRAMES_PATH    = '/content/drive/MyDrive/dataset_frames'  # raw frames
CLASS_NAMES    = ['cheating', 'not_cheating']             # alphabetical (ImageFolder order)
IMG_SIZE       = (224, 224)
BATCH_SIZE     = 32
NUM_EPOCHS     = 20
PATIENCE       = 5

# ── Best model (chosen from Notebook 04 results) ─────────────
# Custom CNN: Acc=0.9780, F1=0.9780, ROC-AUC=0.9995, Params=~200K
# Lightest AND best ROC-AUC → perfect for real-time inference
BEST_MODEL_NAME = 'Custom CNN'
BEST_MODEL_FILE = 'cnn_cheating_model.pth'
BEST_MODEL_KEY  = 'cnn'

# Normalisation for Custom CNN (trained from scratch — NOT ImageNet)
CLF_MEAN = [0.5, 0.5, 0.5]
CLF_STD  = [0.5, 0.5, 0.5]

# All model .pth filenames (for reference / comparison notebook)
MODEL_FILES = {
    'cnn':          'cnn_cheating_model.pth',
    'resnet18':     'resnet18_cheating_model.pth',
    'efficientnet': 'efficientnet_cheating.pth',
    'vit':          'Vision_Transformer.pth',
    'mobilenet':    'mobilenetv2_model.pth',
}

# ── Config (edit these) ──────────────────────────────────────────────
VIDEOS_PATH    = '/content/drive/MyDrive/exam_videos'  # your video folder
OUTPUT_PATH    = FRAMES_PATH                            # = dataset_frames
FRAME_SKIP     = 5        # extract every N-th frame
BLUR_THRESHOLD = 80.0     # Laplacian variance — lower = blurrier; skip if below
IMG_QUALITY    = 95       # JPEG compression quality (0-100)
IMG_EXT        = '.jpg'

# Class sub-folders must exist under VIDEOS_PATH:
#   exam_videos/cheating/     ← put cheating videos here
#   exam_videos/not_cheating/ ← put normal videos here
print(f"📂 Video source  : {VIDEOS_PATH}")
print(f"📂 Frame output  : {OUTPUT_PATH}")
print(f"⚙  Frame skip    : every {FRAME_SKIP} frames")
print(f"⚙  Blur threshold: {BLUR_THRESHOLD}")


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 1 — Validate Folder Structure
# ════════════════════════════════════════════════════════════════════

if not os.path.exists(VIDEOS_PATH):
    raise ValueError(f"❌  Video folder not found: {VIDEOS_PATH}")

found_classes = sorted([
    d for d in os.listdir(VIDEOS_PATH)
    if os.path.isdir(os.path.join(VIDEOS_PATH, d))
])
print(f"📂 Classes found: {found_classes}")

# Warn if class names don't match expected
for cls in CLASS_NAMES:
    if cls not in found_classes:
        print(f"⚠  Expected class folder '{cls}' not found.")

# Create output folders
for cls in found_classes:
    os.makedirs(os.path.join(OUTPUT_PATH, cls), exist_ok=True)

print("✅  Output folders ready.")


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 2 — Extract Frames
# ════════════════════════════════════════════════════════════════════

VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv'}

def is_blurry(frame, threshold):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var() < threshold

total_saved   = 0
total_skipped = 0
summary       = {}

for cls in found_classes:
    cls_video_dir  = os.path.join(VIDEOS_PATH, cls)
    cls_frame_dir  = os.path.join(OUTPUT_PATH, cls)
    cls_saved      = 0
    cls_skipped    = 0
    video_files    = [
        f for f in os.listdir(cls_video_dir)
        if Path(f).suffix.lower() in VIDEO_EXTS
    ]

    print(f"\n📹  [{cls}]  {len(video_files)} video(s) found")

    for video_name in video_files:
        video_path = os.path.join(cls_video_dir, video_name)
        cap        = cv2.VideoCapture(video_path)
        frame_idx  = 0
        saved      = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % FRAME_SKIP == 0:
                if not is_blurry(frame, BLUR_THRESHOLD):
                    out_name  = f"{Path(video_name).stem}_f{frame_idx:06d}{IMG_EXT}"
                    out_path  = os.path.join(cls_frame_dir, out_name)
                    cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, IMG_QUALITY])
                    saved += 1
                else:
                    cls_skipped += 1

            frame_idx += 1

        cap.release()
        cls_saved += saved
        print(f"   {video_name:<40} → {saved} frames saved")

    total_saved   += cls_saved
    total_skipped += cls_skipped
    summary[cls]   = cls_saved
    print(f"   Subtotal: {cls_saved} frames  ({cls_skipped} blurry skipped)")

print(f"\n{'='*50}")
print(f"🎯  TOTAL FRAMES SAVED : {total_saved}")
print(f"🗑️   Blurry frames skipped: {total_skipped}")
print(f"{'='*50}")
for cls, n in summary.items():
    print(f"   {cls:<18}: {n}")
print(f"\n📂  Output: {OUTPUT_PATH}")


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 3 — Quick Sanity Check (display sample frames)
# ════════════════════════════════════════════════════════════════════

import random
import matplotlib.pyplot as plt

IMG_EXTS = {'.jpg', '.jpeg', '.png'}

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()
idx  = 0

for cls in found_classes:
    cls_dir = os.path.join(OUTPUT_PATH, cls)
    images  = [f for f in os.listdir(cls_dir) if Path(f).suffix.lower() in IMG_EXTS]
    samples = random.sample(images, min(4, len(images)))
    for s in samples:
        if idx >= 8: break
        img = cv2.imread(os.path.join(cls_dir, s))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img)
        axes[idx].set_title(cls, fontsize=9)
        axes[idx].axis('off')
        idx += 1

plt.suptitle('Extracted Frame Samples', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()
print("✅  Frame extraction complete. Proceed to Notebook 02.")
